## Setup and Global Configuration

In [ ]:
# !pip install gymnasium[all] matplotlib numpy torch pandas

import math, random, itertools
from collections import defaultdict, deque, namedtuple

import numpy as np
import pandas as pd
import gymnasium as gym
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

FAST_MODE = True  # set False only for longer, higher-precision experiments

if FAST_MODE:
    SEEDS = list(range(30))
    BJ_EPISODES = 1000
    CP_EPISODES = 300
    N_SAMPLES_PER_SA = 50
    DQN_EPISODES = 150
    RAINBOW_EPISODES = 150
else:
    SEEDS = list(range(30))
    BJ_EPISODES = 5000
    CP_EPISODES = 1000
    N_SAMPLES_PER_SA = 300
    DQN_EPISODES = 400
    RAINBOW_EPISODES = 400


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Part 1 — Blackjack

### 1.1 Blackjack MDP and Dynamic Programming

In [ ]:
# Basic Blackjack helpers

CARD_VALUES = [1,2,3,4,5,6,7,8,9,10,10,10,10]
ACTIONS = [0, 1]  # 0=stick, 1=hit

def usable_ace(hand):
    return 1 in hand and sum(hand) + 10 <= 21

def hand_value(hand):
    v = sum(hand)
    if usable_ace(hand):
        return v + 10
    return v

def is_bust(hand):
    return hand_value(hand) > 21

def dealer_policy(hand):
    return hand_value(hand) < 17

def get_state_from_hands(player_hand, dealer_hand):
    return (hand_value(player_hand),
            dealer_hand[0],
            usable_ace(player_hand))

def enumerate_bj_states():
    states = []
    for player_sum in range(4, 32):
        for dealer_show in range(1, 11):
            for ace_flag in [False, True]:
                states.append((player_sum, dealer_show, ace_flag))
    return states

BJ_STATES = enumerate_bj_states()
BJ_STATE_TO_IDX = {s: i for i, s in enumerate(BJ_STATES)}
N_BJ_STATES = len(BJ_STATES)
N_BJ_ACTIONS = len(ACTIONS)
print("Blackjack: states =", N_BJ_STATES, "actions =", N_BJ_ACTIONS)

def bj_draw_card():
    return random.choice(CARD_VALUES)

def reconstruct_player_hand(state):
    player_sum, dealer, ace_flag = state
    if player_sum <= 0:
        return [10, 10]
    if ace_flag:
        raw = player_sum - 10
        if raw <= 1:
            return [1, 3]
        return [1, raw - 1]
    else:
        if player_sum <= 11:
            return [max(1, player_sum - 1), 1]
        return [player_sum - 10, 10]

def simulate_bj_step(state, action):
    player_sum, dealer_show, ace_flag = state
    player_hand = reconstruct_player_hand(state)
    dealer_hand = [dealer_show, bj_draw_card()]

    if action == 1:  # hit
        player_hand.append(bj_draw_card())
        if is_bust(player_hand):
            ns = get_state_from_hands(player_hand, dealer_hand)
            return ns, -1.0, True
        ns = get_state_from_hands(player_hand, dealer_hand)
        return ns, 0.0, False
    else:            # stick
        while dealer_policy(dealer_hand):
            dealer_hand.append(bj_draw_card())
        player_val = hand_value(player_hand)
        dealer_val = hand_value(dealer_hand)
        if is_bust(dealer_hand):
            r = 1.0
        elif dealer_val > player_val:
            r = -1.0
        elif dealer_val < player_val:
            r = 1.0
        else:
            r = 0.0
        ns = get_state_from_hands(player_hand, dealer_hand)
        return ns, r, True

def build_bj_transition_model(n_samples_per_sa=N_SAMPLES_PER_SA, seed=0):
    set_global_seed(seed)
    P = np.zeros((N_BJ_STATES, N_BJ_ACTIONS, N_BJ_STATES), dtype=np.float64)
    R = np.zeros((N_BJ_STATES, N_BJ_ACTIONS, N_BJ_STATES), dtype=np.float64)

    for s_idx, s in enumerate(BJ_STATES):
        ps, dealer, ace = s
        if ps > 21:
            P[s_idx, :, s_idx] = 1.0
            continue
        for a_idx, a in enumerate(ACTIONS):
            counts = defaultdict(int)
            rewards = defaultdict(float)
            for _ in range(n_samples_per_sa):
                ns, r, done = simulate_bj_step(s, a)
                ns_idx = BJ_STATE_TO_IDX.get(ns, s_idx)
                counts[ns_idx] += 1
                rewards[ns_idx] += r
            total = sum(counts.values())
            if total == 0:
                P[s_idx, a_idx, s_idx] = 1.0
            else:
                for ns_idx, c in counts.items():
                    P[s_idx, a_idx, ns_idx] = c / total
                    R[s_idx, a_idx, ns_idx] = rewards[ns_idx] / c
    return P, R

bj_gamma = 1.0
P_bj, R_bj = build_bj_transition_model()
print("P_bj shape:", P_bj.shape)


In [ ]:
# Fast, vectorized Dynamic Programming for tabular MDPs

def value_iteration_fast(P, R, gamma=1.0, theta=1e-2, max_iter=80):
    P = np.asarray(P, dtype=np.float64)
    R = np.asarray(R, dtype=np.float64)

    S, A, _ = P.shape
    V = np.zeros(S, dtype=np.float64)
    delta_hist = []

    for _ in range(max_iter):
        Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        V_new = Q.max(axis=1)
        delta = float(np.max(np.abs(V_new - V)))
        delta_hist.append(delta)
        V = V_new
        if delta < theta:
            break

    Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
    policy = Q.argmax(axis=1).astype(int)
    return V, policy, delta_hist


def policy_evaluation_fast(policy, P, R, gamma=1.0, theta=1e-2, max_iter=50):
    P = np.asarray(P, dtype=np.float64)
    R = np.asarray(R, dtype=np.float64)

    S, A, _ = P.shape
    V = np.zeros(S, dtype=np.float64)

    for _ in range(max_iter):
        Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        V_new = Q[np.arange(S), policy]
        delta = float(np.max(np.abs(V_new - V)))
        V = V_new
        if delta < theta:
            break
    return V


def policy_iteration_fast(P, R, gamma=1.0, theta=1e-2, max_iter=20):
    P = np.asarray(P, dtype=np.float64)
    R = np.asarray(R, dtype=np.float64)

    S, A, _ = P.shape
    policy = np.zeros(S, dtype=int)
    changes_hist = []

    for _ in range(max_iter):
        V = policy_evaluation_fast(policy, P, R, gamma=gamma, theta=theta)
        Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        new_policy = Q.argmax(axis=1).astype(int)

        changes = int(np.sum(new_policy != policy))
        changes_hist.append(changes)
        policy = new_policy

        if changes == 0:
            break

    return V, policy, changes_hist


In [ ]:
# Run DP on Blackjack and plot convergence

V_vi_bj, pi_vi_bj, vi_delta_bj = value_iteration_fast(
    P_bj, R_bj,
    gamma=bj_gamma,
    theta=1e-2,
    max_iter=80
)

V_pi_bj, pi_pi_bj, pi_changes_bj = policy_iteration_fast(
    P_bj, R_bj,
    gamma=bj_gamma,
    theta=1e-2,
    max_iter=20
)

print("Blackjack VI iterations:", len(vi_delta_bj))
print("Blackjack PI iterations:", len(pi_changes_bj))

plt.figure()
plt.plot(vi_delta_bj)
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Max |ΔV|")
plt.title("Blackjack Value Iteration Convergence (fast)")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(pi_changes_bj)
plt.xlabel("Policy iteration")
plt.ylabel("Number of state policy changes")
plt.title("Blackjack Policy Iteration Convergence (fast)")
plt.grid(True)
plt.show()



In [ ]:
# Build 2D grids for value and policy: player_sum (4–21) × dealer_show (1–10)
player_sums = np.arange(4, 22)
dealer_shows = np.arange(1, 11)

def bj_idx(ps, dealer, ace_flag):
    return BJ_STATE_TO_IDX[(ps, dealer, ace_flag)]

V_no_ace = np.zeros((len(player_sums), len(dealer_shows)))
V_ace    = np.zeros((len(player_sums), len(dealer_shows)))
Pi_no_ace = np.zeros_like(V_no_ace)
Pi_ace    = np.zeros_like(V_ace)

for i, ps in enumerate(player_sums):
    for j, dealer in enumerate(dealer_shows):
        s_idx = bj_idx(ps, dealer, False)
        V_no_ace[i, j]  = V_vi_bj[s_idx]
        Pi_no_ace[i, j] = pi_vi_bj[s_idx]
        s_idx = bj_idx(ps, dealer, True)
        V_ace[i, j]  = V_vi_bj[s_idx]
        Pi_ace[i, j] = pi_vi_bj[s_idx]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

im0 = axes[0, 0].imshow(V_no_ace, origin="lower", aspect="auto",
                        extent=[1, 10, 4, 21])
axes[0, 0].set_title("Value (no usable ace)")
axes[0, 0].set_xlabel("Dealer showing")
axes[0, 0].set_ylabel("Player sum")
fig.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(V_ace, origin="lower", aspect="auto",
                        extent=[1, 10, 4, 21])
axes[0, 1].set_title("Value (usable ace)")
axes[0, 1].set_xlabel("Dealer showing")
axes[0, 1].set_ylabel("Player sum")
fig.colorbar(im1, ax=axes[0, 1])

im2 = axes[1, 0].imshow(Pi_no_ace, origin="lower", aspect="auto",
                        extent=[1, 10, 4, 21])
axes[1, 0].set_title("Policy (0=stick,1=hit), no ace")
axes[1, 0].set_xlabel("Dealer showing")
axes[1, 0].set_ylabel("Player sum")
fig.colorbar(im2, ax=axes[1, 0])

im3 = axes[1, 1].imshow(Pi_ace, origin="lower", aspect="auto",
                        extent=[1, 10, 4, 21])
axes[1, 1].set_title("Policy (0=stick,1=hit), usable ace")
axes[1, 1].set_xlabel("Dealer showing")
axes[1, 1].set_ylabel("Player sum")
fig.colorbar(im3, ax=axes[1, 1])

plt.tight_layout()
plt.show()


### 1.2 Blackjack — SARSA vs Q-learning


In [ ]:
def encode_bj_state(state):
    ps, dealer, ace = state
    ps = max(4, min(ps, 31)) - 4      # 0–27
    dealer = dealer - 1               # 0–9
    ace = 1 if ace else 0             # 0–1
    return ps * 20 + dealer * 2 + ace  # roughly 0–1119


def epsilon_greedy_fast(Q, s_idx, n_actions, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(0, n_actions))
    return int(np.argmax(Q[s_idx]))


In [ ]:
def run_sarsa_blackjack_fast(
    num_episodes=BJ_EPISODES,
    alpha=0.1,
    gamma=1.0,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_episodes=None,
    seed=0
):
    env = gym.make("Blackjack-v1", sab=True)
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    n_actions = env.action_space.n
    n_states = 1120  # compact encoded size

    Q = np.zeros((n_states, n_actions), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / eps_decay_episodes)

        state, _ = env.reset(seed=int(rng.integers(1e6)))
        s_idx = encode_bj_state(state)
        action = epsilon_greedy_fast(Q, s_idx, n_actions, eps, rng)

        done = False
        total_reward = 0.0

        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            ns_idx = encode_bj_state(next_state)

            if not done:
                next_action = epsilon_greedy_fast(Q, ns_idx, n_actions, eps, rng)
                target = reward + gamma * Q[ns_idx, next_action]
            else:
                next_action = 0
                target = reward

            td = target - Q[s_idx, action]
            Q[s_idx, action] += alpha * td

            s_idx, action = ns_idx, next_action

        returns[ep] = total_reward

    env.close()
    return Q, returns


def run_q_learning_blackjack_fast(
    num_episodes=BJ_EPISODES,
    alpha=0.1,
    gamma=1.0,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_episodes=None,
    seed=0
):
    env = gym.make("Blackjack-v1", sab=True)
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    n_actions = env.action_space.n
    n_states = 1120
    Q = np.zeros((n_states, n_actions), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / eps_decay_episodes)

        state, _ = env.reset(seed=int(rng.integers(1e6)))
        s_idx = encode_bj_state(state)

        done = False
        total_reward = 0.0

        while not done:
            action = epsilon_greedy_fast(Q, s_idx, n_actions, eps, rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward

            ns_idx = encode_bj_state(next_state)

            if not done:
                target = reward + gamma * np.max(Q[ns_idx])
            else:
                target = reward

            td = target - Q[s_idx, action]
            Q[s_idx, action] += alpha * td

            s_idx = ns_idx

        returns[ep] = total_reward

    env.close()
    return Q, returns


def aggregate_blackjack_fast(algo_fn, seeds, **kwargs):
    rets = []
    for seed in seeds:
        _, r = algo_fn(seed=seed, **kwargs)
        rets.append(r)
    rets = np.stack(rets)
    return np.mean(rets, axis=0), np.std(rets, axis=0)


In [ ]:
sarsa_mean_bj, sarsa_std_bj = aggregate_blackjack_fast(
    run_sarsa_blackjack_fast,
    SEEDS,
    num_episodes=BJ_EPISODES
)

ql_mean_bj, ql_std_bj = aggregate_blackjack_fast(
    run_q_learning_blackjack_fast,
    SEEDS,
    num_episodes=BJ_EPISODES
)

episodes = np.arange(BJ_EPISODES)

plt.figure(figsize=(8,4))
plt.plot(episodes, sarsa_mean_bj, label="SARSA (fast)")
plt.fill_between(episodes, sarsa_mean_bj - sarsa_std_bj, sarsa_mean_bj + sarsa_std_bj, alpha=0.3)
plt.plot(episodes, ql_mean_bj, label="Q-Learning (fast)")
plt.fill_between(episodes, ql_mean_bj - ql_std_bj, ql_mean_bj + ql_std_bj, alpha=0.3)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("Blackjack SARSA vs Q-Learning — Fast Mode")
plt.legend()
plt.grid(True)
plt.show()

# Tabular summary: overall mean and mean over last 200 episodes
window = 200 if BJ_EPISODES >= 200 else BJ_EPISODES

bj_tab = pd.DataFrame([
    {
        "Algorithm": "SARSA",
        "Mean return (all)": float(sarsa_mean_bj.mean()),
        f"Mean return (last {window})": float(sarsa_mean_bj[-window:].mean())
    },
    {
        "Algorithm": "Q-Learning",
        "Mean return (all)": float(ql_mean_bj.mean()),
        f"Mean return (last {window})": float(ql_mean_bj[-window:].mean())
    }
])
bj_tab


In [ ]:
def q_learning_bj_with_dq(
    num_episodes=500,
    alpha=0.1,
    gamma=1.0,
    eps_start=1.0,
    eps_end=0.05,
    seed=0
):
    """Q-learning on Blackjack with logging of max |ΔQ| per episode."""
    env = gym.make("Blackjack-v1", sab=True)
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    n_actions = env.action_space.n
    n_states = 1120
    Q = np.zeros((n_states, n_actions), dtype=np.float32)
    dq_hist = []

    prev_Q = Q.copy()

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / num_episodes)

        state, _ = env.reset(seed=int(rng.integers(1e6)))
        s_idx = encode_bj_state(state)

        done = False
        while not done:
            action = epsilon_greedy_fast(Q, s_idx, n_actions, eps, rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ns_idx = encode_bj_state(next_state)

            if not done:
                target = reward + gamma * np.max(Q[ns_idx])
            else:
                target = reward

            td = target - Q[s_idx, action]
            Q[s_idx, action] += alpha * td
            s_idx = ns_idx

        dq = np.max(np.abs(Q - prev_Q))
        dq_hist.append(dq)
        prev_Q[:] = Q

    env.close()
    return Q, np.array(dq_hist)

Q_dbg, dq_hist = q_learning_bj_with_dq(num_episodes=300, alpha=0.1, eps_end=0.05, seed=0)

plt.figure()
plt.plot(dq_hist)
plt.yscale("log")
plt.xlabel("Episode")
plt.ylabel("max |ΔQ|")
plt.title("Blackjack Q-learning ΔQ over episodes")
plt.grid(True)
plt.show()


# 1.3 Hyperparameter

In [ ]:
# Small hyperparameter sweep for Blackjack Q-learning
alphas = [0.05, 0.1, 0.2]
eps_ends = [0.01, 0.05, 0.1]

rows = []
for a in alphas:
    for e_end in eps_ends:
        mean_ret, _ = aggregate_blackjack_fast(
            run_q_learning_blackjack_fast,
            SEEDS,
            num_episodes=500,
            alpha=a,
            eps_end=e_end
        )
        rows.append({
            "alpha": a,
            "eps_end": e_end,
            "Mean return (all)": float(mean_ret.mean()),
            "Mean return (last 200)": float(mean_ret[-200:].mean())
        })

bj_hp_tab = pd.DataFrame(rows)
bj_hp_tab


## Part 2 — CartPole

### 2.1 Discretized CartPole MDP and Dynamic Programming


In [ ]:
CART_POS_RANGE = (-2.4, 2.4)
CART_VEL_RANGE = (-3.0, 3.0)
POLE_ANG_RANGE = (-0.209, 0.209)
POLE_ANG_VEL_RANGE = (-3.5, 3.5)

CART_BINS = 3
CART_VEL_BINS = 3
POLE_ANG_BINS = 8
POLE_ANG_VEL_BINS = 12

def create_bins(low, high, num_bins):
    return np.linspace(low, high, num_bins + 1)

cart_pos_bins = create_bins(*CART_POS_RANGE, CART_BINS)
cart_vel_bins = create_bins(*CART_VEL_RANGE, CART_VEL_BINS)
pole_ang_bins = create_bins(*POLE_ANG_RANGE, POLE_ANG_BINS)
pole_ang_vel_bins = create_bins(*POLE_ANG_VEL_RANGE, POLE_ANG_VEL_BINS)

def discretize_cartpole_state(state):
    x, x_dot, theta, theta_dot = state

    x = np.clip(x, *CART_POS_RANGE)
    x_dot = np.clip(x_dot, *CART_VEL_RANGE)
    theta = np.clip(theta, *POLE_ANG_RANGE)
    theta_dot = np.clip(theta_dot, *POLE_ANG_VEL_RANGE)

    b0 = np.clip(np.digitize(x, cart_pos_bins) - 1, 0, CART_BINS - 1)
    b1 = np.clip(np.digitize(x_dot, cart_vel_bins) - 1, 0, CART_VEL_BINS - 1)
    b2 = np.clip(np.digitize(theta, pole_ang_bins) - 1, 0, POLE_ANG_BINS - 1)
    b3 = np.clip(np.digitize(theta_dot, pole_ang_vel_bins) - 1, 0, POLE_ANG_VEL_BINS - 1)
    return (b0, b1, b2, b3)

def cartpole_state_index(ds):
    b0, b1, b2, b3 = ds
    return (((b0 * CART_VEL_BINS + b1) * POLE_ANG_BINS + b2) * POLE_ANG_VEL_BINS + b3)

def total_cartpole_states():
    return CART_BINS * CART_VEL_BINS * POLE_ANG_BINS * POLE_ANG_VEL_BINS

N_CP_STATES = total_cartpole_states()
N_CP_ACTIONS = 2

print("CartPole discrete states:", N_CP_STATES, "actions:", N_CP_ACTIONS)

def bin_center(edges, idx):
    return (edges[idx] + edges[idx+1]) / 2.0

def rep_state_from_bins(b0, b1, b2, b3):
    return np.array([
        bin_center(cart_pos_bins, b0),
        bin_center(cart_vel_bins, b1),
        bin_center(pole_ang_bins, b2),
        bin_center(pole_ang_vel_bins, b3)
    ], dtype=np.float32)


In [ ]:
def build_cartpole_transition_model_stochastic(gamma=0.99, base_jitter=0.01, samples=6):
    env = gym.make("CartPole-v1")
    P = np.zeros((N_CP_STATES, N_CP_ACTIONS, N_CP_STATES), dtype=np.float64)
    R = np.zeros((N_CP_STATES, N_CP_ACTIONS, N_CP_STATES), dtype=np.float64)

    for b0 in range(CART_BINS):
        for b1 in range(CART_VEL_BINS):
            for b2 in range(POLE_ANG_BINS):
                for b3 in range(POLE_ANG_VEL_BINS):
                    ds = (b0, b1, b2, b3)
                    s_idx = cartpole_state_index(ds)
                    rep = rep_state_from_bins(b0, b1, b2, b3)

                    for a in range(N_CP_ACTIONS):
                        counts = {}
                        rewards = {}

                        for _ in range(samples):
                            state = rep.copy()
                            if a == 0:
                                state[1] -= base_jitter
                            else:
                                state[1] += base_jitter
                            state += np.random.uniform(-base_jitter/2, base_jitter/2, size=4)

                            env.reset()
                            env.unwrapped.state = state

                            next_state, reward, terminated, truncated, _ = env.step(a)
                            done = terminated or truncated

                            if done:
                                ns_idx = s_idx
                            else:
                                ds_next = discretize_cartpole_state(next_state)
                                ns_idx = cartpole_state_index(ds_next)

                            counts[ns_idx] = counts.get(ns_idx, 0) + 1
                            rewards[ns_idx] = rewards.get(ns_idx, 0.0) + reward

                        total = sum(counts.values())
                        for ns_idx, c in counts.items():
                            P[s_idx, a, ns_idx] = c / total
                            R[s_idx, a, ns_idx] = rewards[ns_idx] / c

    env.close()
    return P, R


def policy_iteration_fast_fixed(P, R, gamma=1.0, theta=1e-3, max_iter=40):
    P = np.asarray(P)
    R = np.asarray(R)

    S, A, _ = P.shape
    policy = np.random.randint(0, A, size=S)
    changes_hist = []

    for it in range(max_iter):
        V = np.zeros(S)
        for _ in range(30):
            Q = (P * (R + gamma * V[None, None, :])).sum(axis=2)
            V_new = Q[np.arange(S), policy]
            if np.max(np.abs(V_new - V)) < theta:
                break
            V = V_new

        Q_full = (P * (R + gamma * V[None, None, :])).sum(axis=2)
        new_policy = Q_full.argmax(axis=1)
        changes = int(np.sum(new_policy != policy))
        changes_hist.append(changes)
        policy = new_policy

        if it >= 3 and changes == 0:
            break

    return V, policy, changes_hist


In [ ]:
cp_gamma = 0.99
P_cp, R_cp = build_cartpole_transition_model_stochastic()
print("P_cp shape:", P_cp.shape)

V_vi_cp, pi_vi_cp, vi_delta_cp = value_iteration_fast(
    P_cp, R_cp, gamma=cp_gamma, theta=1e-3, max_iter=80
)

V_pi_cp, pi_pi_cp, pi_changes_cp = policy_iteration_fast_fixed(
    P_cp, R_cp, gamma=cp_gamma, theta=1e-3, max_iter=40
)

print("CartPole VI iterations:", len(vi_delta_cp))
print("CartPole PI iterations:", len(pi_changes_cp))

plt.figure()
plt.plot(vi_delta_cp)
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Max |ΔV|")
plt.title("CartPole Value Iteration Convergence (fast)")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(pi_changes_cp)
plt.xlabel("Policy iteration")
plt.ylabel("# state policy changes")
plt.title("CartPole Policy Iteration Convergence (fast, stochastic MDP)")
plt.grid(True)
plt.show()



### 2.2 CartPole — SARSA vs Q-learning



In [ ]:
def run_sarsa_cartpole(
    num_episodes=CP_EPISODES,
    alpha=0.5,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.01,
    eps_decay_episodes=None,
    seed=0
):
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    Q = np.zeros((N_CP_STATES, N_CP_ACTIONS), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(
            eps_end,
            eps_start - (eps_start - eps_end) * ep / max(1, eps_decay_episodes)
        )

        obs, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ds = discretize_cartpole_state(obs)
        s_idx = cartpole_state_index(ds)
        action = epsilon_greedy_fast(Q, s_idx, N_CP_ACTIONS, eps, rng)
        done = False
        ep_return = 0.0

        while not done:
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward

            if not done:
                ds_next = discretize_cartpole_state(next_obs)
                ns_idx = cartpole_state_index(ds_next)
                next_action = epsilon_greedy_fast(Q, ns_idx, N_CP_ACTIONS, eps, rng)

                td_target = reward + gamma * Q[ns_idx, next_action]
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error

                s_idx, action = ns_idx, next_action
            else:
                td_target = reward
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error

        returns[ep] = ep_return

    env.close()
    return Q, returns


def run_q_learning_cartpole(
    num_episodes=CP_EPISODES,
    alpha=0.5,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.01,
    eps_decay_episodes=None,
    seed=0
):
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    if eps_decay_episodes is None:
        eps_decay_episodes = num_episodes

    Q = np.zeros((N_CP_STATES, N_CP_ACTIONS), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    for ep in range(num_episodes):
        eps = max(
            eps_end,
            eps_start - (eps_start - eps_end) * ep / max(1, eps_decay_episodes)
        )

        obs, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ds = discretize_cartpole_state(obs)
        s_idx = cartpole_state_index(ds)
        done = False
        ep_return = 0.0

        while not done:
            action = epsilon_greedy_fast(Q, s_idx, N_CP_ACTIONS, eps, rng)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward

            if not done:
                ds_next = discretize_cartpole_state(next_obs)
                ns_idx = cartpole_state_index(ds_next)
                best_next = np.max(Q[ns_idx])

                td_target = reward + gamma * best_next
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error

                s_idx = ns_idx
            else:
                td_target = reward
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error

        returns[ep] = ep_return

    env.close()
    return Q, returns


def aggregate_cartpole(algo_fn, seeds, **kwargs):
    all_returns = []
    for seed in seeds:
        _, ret = algo_fn(seed=seed, **kwargs)
        all_returns.append(ret)
    arr = np.stack(all_returns)
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)
    return mean, std


In [ ]:
sarsa_mean_cp, sarsa_std_cp = aggregate_cartpole(
    run_sarsa_cartpole,
    SEEDS,
    num_episodes=CP_EPISODES,
    alpha=0.5,
    gamma=0.99
)

ql_mean_cp, ql_std_cp = aggregate_cartpole(
    run_q_learning_cartpole,
    SEEDS,
    num_episodes=CP_EPISODES,
    alpha=0.5,
    gamma=0.99
)

episodes_cp = np.arange(CP_EPISODES)

plt.figure()
plt.plot(episodes_cp, sarsa_mean_cp, label="SARSA")
plt.fill_between(episodes_cp, sarsa_mean_cp - sarsa_std_cp, sarsa_mean_cp + sarsa_std_cp, alpha=0.3)
plt.plot(episodes_cp, ql_mean_cp, label="Q-Learning")
plt.fill_between(episodes_cp, ql_mean_cp - ql_std_cp, ql_mean_cp + ql_std_cp, alpha=0.3)
plt.xlabel("Episode")
plt.ylabel("Episode length (return)")
plt.title("CartPole: SARSA vs Q-Learning (mean ± std over seeds)")
plt.legend()
plt.grid(True)
plt.show()

cp_window = 100 if CP_EPISODES >= 100 else CP_EPISODES

cp_tab = pd.DataFrame([
    {
        "Algorithm": "SARSA",
        "Mean return (all)": float(sarsa_mean_cp.mean()),
        f"Mean return (last {cp_window})": float(sarsa_mean_cp[-cp_window:].mean())
    },
    {
        "Algorithm": "Q-Learning",
        "Mean return (all)": float(ql_mean_cp.mean()),
        f"Mean return (last {cp_window})": float(ql_mean_cp[-cp_window:].mean())
    }
])
cp_tab


## 2.3 Hyperparameters

In [ ]:
def make_cartpole_bins(config):
    """Return bin arrays for a given (cart, cart_vel, pole, pole_vel) bin configuration."""
    c_bins, cv_bins, p_bins, pv_bins = config
    cart_pos_bins = create_bins(*CART_POS_RANGE, c_bins)
    cart_vel_bins = create_bins(*CART_VEL_RANGE, cv_bins)
    pole_ang_bins = create_bins(*POLE_ANG_RANGE, p_bins)
    pole_ang_vel_bins = create_bins(*POLE_ANG_VEL_RANGE, pv_bins)
    return cart_pos_bins, cart_vel_bins, pole_ang_bins, pole_ang_vel_bins

def discretize_cartpole_state_cfg(state, bins):
    """Discretize CartPole state given a particular bin configuration."""
    cart_pos_bins, cart_vel_bins, pole_ang_bins, pole_ang_vel_bins = bins
    x, x_dot, theta, theta_dot = state
    x = np.clip(x, *CART_POS_RANGE)
    x_dot = np.clip(x_dot, *CART_VEL_RANGE)
    theta = np.clip(theta, *POLE_ANG_RANGE)
    theta_dot = np.clip(theta_dot, *POLE_ANG_VEL_RANGE)

    b0 = np.clip(np.digitize(x, cart_pos_bins) - 1, 0, len(cart_pos_bins)-2)
    b1 = np.clip(np.digitize(x_dot, cart_vel_bins) - 1, 0, len(cart_vel_bins)-2)
    b2 = np.clip(np.digitize(theta, pole_ang_bins) - 1, 0, len(pole_ang_bins)-2)
    b3 = np.clip(np.digitize(theta_dot, pole_ang_vel_bins) - 1, 0, len(pole_ang_vel_bins)-2)
    return (b0, b1, b2, b3)

def run_q_learning_cartpole_cfg(
    bins,
    num_episodes=200,
    alpha=0.5,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.01,
    seed=0
):
    """Q-learning on CartPole for a given discretization configuration."""
    c_bins, cv_bins, p_bins, pv_bins = bins
    n_states = (len(c_bins)-1)*(len(cv_bins)-1)*(len(p_bins)-1)*(len(pv_bins)-1)
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)
    Q = np.zeros((n_states, N_CP_ACTIONS), dtype=np.float32)
    returns = np.zeros(num_episodes, dtype=np.float32)

    def idx(ds):
        b0, b1, b2, b3 = ds
        return (((b0 * (len(cv_bins)-1) + b1) * (len(p_bins)-1) + b2) * (len(pv_bins)-1) + b3)

    for ep in range(num_episodes):
        eps = max(eps_end, eps_start - (eps_start - eps_end) * ep / num_episodes)
        obs, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ds = discretize_cartpole_state_cfg(obs, bins)
        s_idx = idx(ds)
        done = False
        ep_return = 0.0
        while not done:
            action = epsilon_greedy_fast(Q, s_idx, N_CP_ACTIONS, eps, rng)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_return += reward
            if not done:
                ds_next = discretize_cartpole_state_cfg(next_obs, bins)
                ns_idx = idx(ds_next)
                best_next = np.max(Q[ns_idx])
                td_target = reward + gamma * best_next
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error
                s_idx = ns_idx
            else:
                td_target = reward
                td_error = td_target - Q[s_idx, action]
                Q[s_idx, action] += alpha * td_error
        returns[ep] = ep_return

    env.close()
    return returns

configs = [
    (2, 2, 6, 8),
    (3, 3, 8, 12),   # main config
    (4, 4, 10, 14)
]

rows = []
for cfg in configs:
    bins = make_cartpole_bins(cfg)
    rets = run_q_learning_cartpole_cfg(bins, num_episodes=200, seed=0)
    rows.append({
        "cart_bins": cfg[0],
        "cart_vel_bins": cfg[1],
        "pole_bins": cfg[2],
        "pole_vel_bins": cfg[3],
        "Mean return (all)": float(rets.mean()),
        "Mean return (last 50)": float(rets[-50:].mean())
    })

disc_tab = pd.DataFrame(rows)
disc_tab


## Part 3 — Extra Credit: DQN and Rainbow-lite on CartPole


### 3.1 Replay Buffers and Q-Networks

In [ ]:
Transition = namedtuple("Transition", ("state", "action", "reward", "next_state", "done"))

class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)


class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.pos = 0
        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.eps = 1e-5

    def push(self, *args):
        max_prio = self.priorities.max() if self.buffer else 1.0
        if len(self.buffer) < self.capacity:
            self.buffer.append(Transition(*args))
        else:
            self.buffer[self.pos] = Transition(*args)
        self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        if len(self.buffer) == self.capacity:
            prios = self.priorities
        else:
            prios = self.priorities[:self.pos]
        probs = prios ** self.alpha
        probs /= probs.sum()
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]

        total = len(self.buffer)
        weights = (total * probs[indices]) ** (-beta)
        weights /= weights.max()
        weights = torch.tensor(weights, dtype=torch.float32, device=device)

        batch = Transition(*zip(*samples))
        return batch, indices, weights

    def update_priorities(self, indices, priorities):
        for idx, prio in zip(indices, priorities):
            self.priorities[idx] = float(prio + self.eps)

    def __len__(self):
        return len(self.buffer)


class DQNNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)


class DuelingDQNNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU()
        )
        self.advantage = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        self.value = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        f = self.feature(x)
        adv = self.advantage(f)
        val = self.value(f)
        q = val + (adv - adv.mean(dim=1, keepdim=True))
        return q


### 3.2 DQN Agent and Training

In [ ]:
class DQNAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000
    ):
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay_steps = eps_decay_steps

        self.online_net = DQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net = DQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_capacity)

        self.total_steps = 0

    def epsilon(self):
        return max(
            self.eps_end,
            self.eps_start - (self.eps_start - self.eps_end) * (self.total_steps / self.eps_decay_steps)
        )

    def select_action(self, state, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        if rng.random() < self.epsilon():
            return int(rng.integers(0, self.action_dim))
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_vals = self.online_net(state_t)
        return int(q_vals.argmax(dim=1).item())

    def push(self, *args):
        self.buffer.push(*args)

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None
        transitions = self.buffer.sample(self.batch_size)
        states = torch.tensor(np.array(transitions.state), dtype=torch.float32, device=device)
        actions = torch.tensor(transitions.action, dtype=torch.int64, device=device).unsqueeze(-1)
        rewards = torch.tensor(transitions.reward, dtype=torch.float32, device=device).unsqueeze(-1)
        next_states = torch.tensor(np.array(transitions.next_state), dtype=torch.float32, device=device)
        dones = torch.tensor(transitions.done, dtype=torch.float32, device=device).unsqueeze(-1)

        q_vals = self.online_net(states).gather(1, actions)
        with torch.no_grad():
            max_next = self.target_net(next_states).max(dim=1, keepdim=True)[0]
            target = rewards + self.gamma * (1.0 - dones) * max_next
        loss = nn.MSELoss()(q_vals, target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.total_steps % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        return loss.item()


def train_dqn_cartpole(
    num_episodes=DQN_EPISODES,
    max_steps_per_ep=500,
    seed=0
):
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = DQNAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000
    )

    episode_returns = []
    global_step = 0

    for ep in range(num_episodes):
        state, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ep_return = 0.0

        for t in range(max_steps_per_ep):
            agent.total_steps = global_step
            action = agent.select_action(state, rng=rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.push(state, action, reward, next_state, float(done))
            agent.update()

            state = next_state
            ep_return += reward
            global_step += 1

            if done:
                break

        episode_returns.append(ep_return)
        if (ep + 1) % 50 == 0:
            print(f"[DQN] Episode {ep+1}/{num_episodes}, return={ep_return:.1f}")

    env.close()
    return episode_returns


### 3.3 Rainbow-lite Agent and Training

In [ ]:
class RainbowLiteAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        per_beta_steps=50_000
    ):
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update

        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay_steps = eps_decay_steps

        self.per_alpha = per_alpha
        self.per_beta_start = per_beta_start
        self.per_beta_end = per_beta_end
        self.per_beta_steps = per_beta_steps

        self.online_net = DuelingDQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net = DuelingDQNNet(state_dim, action_dim, hidden_dim).to(device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        self.buffer = PrioritizedReplayBuffer(buffer_capacity, alpha=per_alpha)

        self.total_steps = 0

    def epsilon(self):
        return max(
            self.eps_end,
            self.eps_start - (self.eps_start - self.eps_end) * (self.total_steps / self.eps_decay_steps)
        )

    def beta(self):
        return min(
            self.per_beta_end,
            self.per_beta_start + (self.per_beta_end - self.per_beta_start) * (self.total_steps / self.per_beta_steps)
        )

    def select_action(self, state, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        if rng.random() < self.epsilon():
            return int(rng.integers(0, self.action_dim))
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_vals = self.online_net(state_t)
        return int(q_vals.argmax(dim=1).item())

    def push(self, *args):
        self.buffer.push(*args)

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None
        beta = self.beta()
        transitions, indices, weights = self.buffer.sample(self.batch_size, beta=beta)

        states = torch.tensor(np.array(transitions.state), dtype=torch.float32, device=device)
        actions = torch.tensor(transitions.action, dtype=torch.int64, device=device).unsqueeze(-1)
        rewards = torch.tensor(transitions.reward, dtype=torch.float32, device=device).unsqueeze(-1)
        next_states = torch.tensor(np.array(transitions.next_state), dtype=torch.float32, device=device)
        dones = torch.tensor(transitions.done, dtype=torch.float32, device=device).unsqueeze(-1)

        q_vals = self.online_net(states).gather(1, actions)

        with torch.no_grad():
            next_online = self.online_net(next_states)
            best_actions = next_online.argmax(dim=1, keepdim=True)
            next_target = self.target_net(next_states)
            next_q = next_target.gather(1, best_actions)
            target = rewards + self.gamma * (1.0 - dones) * next_q

        td_errors = target - q_vals
        loss = (weights.unsqueeze(-1) * td_errors.pow(2)).mean()

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        new_prios = td_errors.detach().abs().squeeze(-1).cpu().numpy()
        self.buffer.update_priorities(indices, new_prios)

        if self.total_steps % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())
        return loss.item()


def train_rainbow_lite_cartpole(
    num_episodes=RAINBOW_EPISODES,
    max_steps_per_ep=500,
    seed=0
):
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = RainbowLiteAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=128,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000,
        per_alpha=0.6,
        per_beta_start=0.4,
        per_beta_end=1.0,
        per_beta_steps=50_000
    )

    episode_returns = []
    global_step = 0

    for ep in range(num_episodes):
        state, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ep_return = 0.0

        for t in range(max_steps_per_ep):
            agent.total_steps = global_step
            action = agent.select_action(state, rng=rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.push(state, action, reward, next_state, float(done))
            agent.update()

            state = next_state
            ep_return += reward
            global_step += 1

            if done:
                break

        episode_returns.append(ep_return)
        if (ep + 1) % 50 == 0:
            print(f"[Rainbow-lite] Episode {ep+1}/{num_episodes}, return={ep_return:.1f}")

    env.close()
    return episode_returns


### 3.4 DQN vs Rainbow-lite — Training and Summary

In [ ]:
dqn_returns = train_dqn_cartpole(num_episodes=DQN_EPISODES, max_steps_per_ep=500, seed=30)
rainbow_returns = train_rainbow_lite_cartpole(num_episodes=RAINBOW_EPISODES, max_steps_per_ep=500, seed=30)

episodes = np.arange(len(dqn_returns))

plt.figure()
plt.plot(episodes, dqn_returns, label="DQN")
plt.plot(episodes, rainbow_returns, label="Rainbow-lite")
plt.xlabel("Episode")
plt.ylabel("Return (episode length)")
plt.title("CartPole: DQN vs Rainbow-lite (extra credit)")
plt.legend()
plt.grid(True)
plt.show()

# Final tabular summary for extra credit
ec_window = 20 if DQN_EPISODES >= 20 else DQN_EPISODES

extra_tab = pd.DataFrame([
    {
        "Agent": "DQN",
        "Mean return (all)": float(np.mean(dqn_returns)),
        f"Mean return (last {ec_window})": float(np.mean(dqn_returns[-ec_window:]))
    },
    {
        "Agent": "Rainbow-lite",
        "Mean return (all)": float(np.mean(rainbow_returns)),
        f"Mean return (last {ec_window})": float(np.mean(rainbow_returns[-ec_window:]))
    },
])
extra_tab


## 3.5 DQN / Rainbow-lite ablations

In [ ]:
def train_dqn_cartpole_variant(hidden_dim=64, num_episodes=100, seed=40):
    """
    DQN training with a configurable hidden layer size to study capacity effects.
    """
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = DQNAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=hidden_dim,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000
    )

    episode_returns = []
    global_step = 0
    for ep in range(num_episodes):
        state, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ep_return = 0.0
        for t in range(500):
            agent.total_steps = global_step
            action = agent.select_action(state, rng=rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.push(state, action, reward, next_state, float(done))
            agent.update()
            state = next_state
            ep_return += reward
            global_step += 1
            if done:
                break
        episode_returns.append(ep_return)

    env.close()
    return np.array(episode_returns)

dqn_small = train_dqn_cartpole_variant(hidden_dim=64, num_episodes=100, seed=30)
dqn_big   = train_dqn_cartpole_variant(hidden_dim=256, num_episodes=100, seed=30)

abl_tab = pd.DataFrame([
    {
        "Variant": "DQN-64",
        "Mean return (all)": float(dqn_small.mean()),
        "Mean return (last 20)": float(dqn_small[-20:].mean())
    },
    {
        "Variant": "DQN-256",
        "Mean return (all)": float(dqn_big.mean()),
        "Mean return (last 20)": float(dqn_big[-20:].mean())
    },
])
abl_tab


In [ ]:
def train_rainbow_cartpole_variant(
    hidden_dim=128,
    per_alpha=0.6,
    num_episodes=100,
    seed=40
):
    """
    Rainbow-lite training with configurable:
      - hidden_dim (network capacity)
      - per_alpha (strength of prioritized experience replay)
    """
    env = gym.make("CartPole-v1")
    set_global_seed(seed)
    rng = np.random.default_rng(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = RainbowLiteAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dim=hidden_dim,
        gamma=0.99,
        lr=1e-3,
        batch_size=64,
        buffer_capacity=50_000,
        target_update=1000,
        eps_start=1.0,
        eps_end=0.01,
        eps_decay_steps=50_000,
        per_alpha=per_alpha,        # <-- Ablation variable
        per_beta_start=0.4,
        per_beta_end=1.0,
        per_beta_steps=50_000
    )

    episode_returns = []
    global_step = 0

    for ep in range(num_episodes):
        state, _ = env.reset(seed=int(rng.integers(0, 10_000)))
        ep_return = 0.0

        for t in range(500):
            agent.total_steps = global_step
            action = agent.select_action(state, rng=rng)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.push(state, action, reward, next_state, float(done))
            agent.update()

            state = next_state
            ep_return += reward
            global_step += 1

            if done:
                break

        episode_returns.append(ep_return)

    env.close()
    return np.array(episode_returns)

# Rainbow-lite variants
rainbow_small_net = train_rainbow_cartpole_variant(
    hidden_dim=64, per_alpha=0.6, num_episodes=100, seed=30
)

rainbow_high_per = train_rainbow_cartpole_variant(
    hidden_dim=128, per_alpha=0.9, num_episodes=100, seed=30
)


rainbow_abl_tab = pd.DataFrame([
    {
        "Variant": "Rainbow-64 (small net)",
        "Mean return (all)": float(rainbow_small_net.mean()),
        "Mean return (last 20)": float(rainbow_small_net[-20:].mean())
    },
    {
        "Variant": "Rainbow α=0.9 (strong PER)",
        "Mean return (all)": float(rainbow_high_per.mean()),
        "Mean return (last 20)": float(rainbow_high_per[-20:].mean())
    }
])

rainbow_abl_tab


# 4. Reproducibility (40-seed runs)

In [ ]:
# Extra set of seeds used only for the reproducibility section
REPRO_SEEDS = list(range(40))

## Reproducibility: 40-seed evaluation for tabular methods

def run_blackjack_with_seeds_40():
    """
    Re-run Blackjack tabular methods (SARSA and Q-learning)
    over 40 random seeds to report mean ± std performance.
    """
    # SARSA
    sarsa_rets = []
    for seed in REPRO_SEEDS:
        _, r = run_sarsa_blackjack_fast(
            num_episodes=BJ_EPISODES,
            alpha=0.1,
            gamma=1.0,
            eps_start=1.0,
            eps_end=0.05,
            seed=seed
        )
        sarsa_rets.append(r)
    sarsa_rets = np.stack(sarsa_rets)
    sarsa_mean_all = float(sarsa_rets.mean())
    sarsa_mean_last = float(sarsa_rets[:, -200:].mean()) if BJ_EPISODES >= 200 else float(sarsa_rets.mean())
    sarsa_std_last = float(sarsa_rets[:, -200:].mean(axis=1).std())

    # Q-learning
    ql_rets = []
    for seed in REPRO_SEEDS:
        _, r = run_q_learning_blackjack_fast(
            num_episodes=BJ_EPISODES,
            alpha=0.1,
            gamma=1.0,
            eps_start=1.0,
            eps_end=0.05,
            seed=seed
        )
        ql_rets.append(r)
    ql_rets = np.stack(ql_rets)
    ql_mean_all = float(ql_rets.mean())
    ql_mean_last = float(ql_rets[:, -200:].mean()) if BJ_EPISODES >= 200 else float(ql_rets.mean())
    ql_std_last = float(ql_rets[:, -200:].mean(axis=1).std())

    bj_repro_tab = pd.DataFrame([
        {
            "Env": "Blackjack",
            "Algorithm": "SARSA",
            "Seeds": len(REPRO_SEEDS),
            "Mean return (all episodes)": sarsa_mean_all,
            "Mean return (last 200)": sarsa_mean_last,
            "Std over seeds (last 200)": sarsa_std_last,
        },
        {
            "Env": "Blackjack",
            "Algorithm": "Q-Learning",
            "Seeds": len(REPRO_SEEDS),
            "Mean return (all episodes)": ql_mean_all,
            "Mean return (last 200)": ql_mean_last,
            "Std over seeds (last 200)": ql_std_last,
        },
    ])
    return bj_repro_tab


def run_cartpole_with_seeds_40():
    """
    Re-run CartPole tabular methods (SARSA and Q-learning)
    over 40 random seeds to report mean ± std performance.
    """
    # SARSA
    sarsa_rets = []
    for seed in REPRO_SEEDS:
        _, r = run_sarsa_cartpole(
            num_episodes=CP_EPISODES,
            alpha=0.5,
            gamma=0.99,
            eps_start=1.0,
            eps_end=0.01,
            seed=seed
        )
        sarsa_rets.append(r)
    sarsa_rets = np.stack(sarsa_rets)
    sarsa_mean_all = float(sarsa_rets.mean())
    sarsa_mean_last = float(sarsa_rets[:, -100:].mean()) if CP_EPISODES >= 100 else float(sarsa_rets.mean())
    sarsa_std_last = float(sarsa_rets[:, -100:].mean(axis=1).std())

    # Q-learning
    ql_rets = []
    for seed in REPRO_SEEDS:
        _, r = run_q_learning_cartpole(
            num_episodes=CP_EPISODES,
            alpha=0.5,
            gamma=0.99,
            eps_start=1.0,
            eps_end=0.01,
            seed=seed
        )
        ql_rets.append(r)
    ql_rets = np.stack(ql_rets)
    ql_mean_all = float(ql_rets.mean())
    ql_mean_last = float(ql_rets[:, -100:].mean()) if CP_EPISODES >= 100 else float(ql_rets.mean())
    ql_std_last = float(ql_rets[:, -100:].mean(axis=1).std())

    cp_repro_tab = pd.DataFrame([
        {
            "Env": "CartPole",
            "Algorithm": "SARSA",
            "Seeds": len(REPRO_SEEDS),
            "Mean return (all episodes)": sarsa_mean_all,
            "Mean return (last 100)": sarsa_mean_last,
            "Std over seeds (last 100)": sarsa_std_last,
        },
        {
            "Env": "CartPole",
            "Algorithm": "Q-Learning",
            "Seeds": len(REPRO_SEEDS),
            "Mean return (all episodes)": ql_mean_all,
            "Mean return (last 100)": ql_mean_last,
            "Std over seeds (last 100)": ql_std_last,
        },
    ])
    return cp_repro_tab


# Run reproducibility experiments and display summary tables

bj_repro_tab = run_blackjack_with_seeds_40()
cp_repro_tab = run_cartpole_with_seeds_40()

print("=== Reproducibility summary: 40 seeds (Blackjack) ===")
display(bj_repro_tab)

print("\n=== Reproducibility summary: 40 seeds (CartPole) ===")
display(cp_repro_tab)
